In [1]:
import cv2
import numpy as np
import mediapipe as mp
import pickle
import tensorflow as tf
from collections import deque, Counter

# =========================
# PATH MODEL
# =========================
MODEL_PATH = "model_angka_bilstm_attention.keras"
ENCODER_PATH = "label_encoder_angka_video.pkl"

SEQUENCE_LENGTH = 30
FEATURE_SIZE = 63
CONFIDENCE_THRESHOLD = 0.7

# =========================
# LOAD MODEL
# =========================
model = tf.keras.models.load_model(MODEL_PATH)

with open(ENCODER_PATH, "rb") as f:
    label_encoder = pickle.load(f)

print("Classes:", label_encoder.classes_)

# =========================
# MEDIAPIPE
# =========================
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# =========================
# NORMALISASI LANDMARK
# =========================
def normalize_landmarks(hand_landmarks):
    landmarks = []

    for lm in hand_landmarks.landmark:
        landmarks.append([lm.x, lm.y, lm.z])

    landmarks = np.array(landmarks)

    wrist = landmarks[0]
    landmarks = landmarks - wrist

    scale = np.linalg.norm(landmarks[9])

    if scale < 1e-6:
        scale = 1.0

    landmarks = landmarks / scale

    return landmarks.flatten()

# =========================
# REALTIME
# =========================
cap = cv2.VideoCapture(0)

sequence = deque(maxlen=SEQUENCE_LENGTH)
predictions_buffer = deque(maxlen=10)

final_prediction = "-"
final_confidence = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    landmark_data = np.zeros(FEATURE_SIZE)

    if result.multi_hand_landmarks:
        hand_landmarks = result.multi_hand_landmarks[0]

        mp_draw.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS
        )

        landmark_data = normalize_landmarks(hand_landmarks)

    sequence.append(landmark_data)

    if len(sequence) == SEQUENCE_LENGTH:
        input_data = np.expand_dims(np.array(sequence), axis=0)

        pred = model.predict(input_data, verbose=0)
        pred_index = np.argmax(pred)
        confidence = np.max(pred)

        pred_label = label_encoder.inverse_transform([pred_index])[0]

        if confidence >= CONFIDENCE_THRESHOLD:
            predictions_buffer.append(pred_label)

            most_common = Counter(predictions_buffer).most_common(1)[0][0]
            final_prediction = most_common
            final_confidence = confidence

    cv2.putText(frame, f"Prediksi: {final_prediction}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.putText(frame, f"Confidence: {final_confidence:.2f}", (30, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, "Tekan Q untuk keluar", (30, 135),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow("Realtime Angka Video", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'tensorflow'